# Forest Fire Detection — Final Image Model Training
## Notebook 04: Full Training of Selected Model with Best Practices

Transfer learning with early stopping, LR scheduling, class weighting, and checkpointing.


In [ ]:
import os, sys, json, time, copy
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
META_DIR = IMPL / "artifacts" / "metadata"
PLOTS    = IMPL / "artifacts" / "plots"
MODEL_DIR = IMPL / "models" / "image"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Load config
with open(META_DIR / "data_splits.json") as f:
    splits = json.load(f)
with open(META_DIR / "class_weights.json") as f:
    cw_data = json.load(f)
with open(META_DIR / "selected_model.json") as f:
    sel = json.load(f)

SELECTED_MODEL = sel['model_name']
CLASS_NAMES    = cw_data['class_names']
CLASS_TO_IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS   = {i: c for i, c in enumerate(CLASS_NAMES)}
cw             = cw_data['class_weights']

print(f"Device:         {DEVICE}")
print(f"Selected model: {SELECTED_MODEL}")
print(f"Classes:        {CLASS_NAMES}")


In [ ]:
IMAGE_SIZE    = 224
BATCH_SIZE    = 16
LEARNING_RATE = 1e-3
MAX_EPOCHS    = 20
PATIENCE      = 5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class FireDataset(torch.utils.data.Dataset):
    def __init__(self, data, transform=None):
        self.data      = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, CLASS_TO_IDX[label]

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=12),
    T.ColorJitter(brightness=0.25, contrast=0.2, saturation=0.1, hue=0.05),
    T.RandomGrayscale(p=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_ds = FireDataset(splits['train'], train_transform)
val_ds   = FireDataset(splits['val'],   val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

weight_tensor = torch.tensor([cw['FIRE'], cw['NO_FIRE']], dtype=torch.float32).to(DEVICE)

print(f"Training samples:   {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Batch size: {BATCH_SIZE} | Max epochs: {MAX_EPOCHS} | Patience: {PATIENCE}")


In [ ]:
def build_final_model(model_name, num_classes=2, fine_tune_layers=2):
    if model_name == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        # Unfreeze last N feature blocks + classifier
        feature_blocks = list(m.features.children())
        for block in feature_blocks[:-fine_tune_layers]:
            for p in block.parameters():
                p.requires_grad = False
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'MobileNetV3':
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        feature_blocks = list(m.features.children())
        for block in feature_blocks[:-fine_tune_layers]:
            for p in block.parameters():
                p.requires_grad = False
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif model_name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        feature_blocks = list(m.features.children())
        for block in feature_blocks[:-fine_tune_layers]:
            for p in block.parameters():
                p.requires_grad = False
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for name, p in m.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(DEVICE)

model = build_final_model(SELECTED_MODEL)
n_params     = sum(p.numel() for p in model.parameters())
n_trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {SELECTED_MODEL}")
print(f"  Total params:     {n_params:,}")
print(f"  Trainable params: {n_trainable:,}")


In [ ]:
criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}
best_val_f1  = 0.0
best_weights = None
no_improve   = 0

print(f"Starting training...")
t_start = time.time()

for epoch in range(MAX_EPOCHS):
    # ── Train ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    train_correct = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (out.argmax(1) == lbls).sum().item()
    
    train_loss /= len(train_loader)
    train_acc   = train_correct / len(train_ds)
    
    # ── Validate ────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    all_preds, all_lbls = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, lbls)
            val_loss += loss.item()
            preds = out.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_lbls.extend(lbls.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_acc   = accuracy_score(all_lbls, all_preds)
    val_f1    = f1_score(all_lbls, all_preds, average='weighted', zero_division=0)
    
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(lr)
    
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
          f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | "
          f"lr={lr:.2e}")
    
    # ── Early stopping ─────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_weights = copy.deepcopy(model.state_dict())
        no_improve   = 0
        torch.save(best_weights, MODEL_DIR / "best_model.pth")
        print(f"  ✓ New best val_f1={best_val_f1:.4f} — checkpoint saved")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

train_time = time.time() - t_start
print(f"\nTraining complete in {train_time:.1f}s ({train_time/60:.1f} min)")
print(f"Best validation F1: {best_val_f1:.4f}")


In [ ]:
# Load best weights
model.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location=DEVICE))
model.eval()

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_ran = list(range(1, len(history['train_loss'])+1))

axes[0].plot(epochs_ran, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   'r-s', label='Val')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   'r-s', label='Val')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history['val_f1'], 'g-^', label='Val F1')
ax2 = axes[2].twinx()
ax2.plot(epochs_ran, history['lr'], 'k--', label='LR', alpha=0.5)
axes[2].set_title('Val F1 & Learning Rate', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS / "training_curves.png", dpi=100, bbox_inches='tight')
plt.close()
print("Training curves saved.")


In [ ]:
# Save class names and metadata
class_names_data = {'class_names': CLASS_NAMES, 'class_to_idx': CLASS_TO_IDX, 'idx_to_class': IDX_TO_CLASS}
with open(MODEL_DIR / "class_names.json", "w") as f:
    json.dump(class_names_data, f, indent=2)

model_metadata = {
    'model_name': SELECTED_MODEL,
    'image_size': IMAGE_SIZE,
    'num_classes': 2,
    'class_names': CLASS_NAMES,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'best_val_f1': round(best_val_f1, 4),
    'training_time_s': round(train_time, 1),
    'epochs_ran': len(history['train_loss']),
    'device': str(DEVICE)
}
with open(MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

# Save history
with open(META_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Model artifacts saved:")
print(f"  {MODEL_DIR}/best_model.pth")
print(f"  {MODEL_DIR}/class_names.json")
print(f"  {MODEL_DIR}/model_metadata.json")
print(f"\nFinal model: {SELECTED_MODEL}")
print(f"Best Val F1: {best_val_f1:.4f}")
